<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
<br/>
👉 <b>Note:</b> This Colab notebook illustrates simplified usage of <code>rapidfireai</code>. For the full RapidFire AI experience with advanced experiment manager, UI, and production features, see our <a href="https://oss-docs.rapidfire.ai/en/latest/walkthrough.html">Install and Get Started</a> guide.
<br/>
🎬 Watch our <a href="https://youtu.be/nPMBfZWqPWI">intro video</a> to get started!

</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RapidFireAI/rapidfireai/blob/main/community_notebooks/sft-child-facing-chatbot.ipynb)

⚠️ **IMPORTANT:** Do not let the Colab notebook tab stay idle for more than 5 min; Colab will disconnect otherwise. Refresh the TensorBoard screen or interact with the cells to avoid disconnection.

# Supervised Fine-Tuning (SFT) for a Child-Facing Chatbot

## What We're Building

We'll fine-tune a **child-facing conversational chatbot** using [TinyLlama-1.1B-Chat](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0), a compact 1.1B-parameter chat model. The chatbot is trained to produce **age-appropriate, supportive, and instruction-aligned responses** to children's questions.

The training dataset was **self-generated using the Gemini API**, synthesized by combining labels and children's speech texts from two Hugging Face children-focused datasets, then augmented with age-appropriate, label-consistent responses generated by Gemini. The dataset is available at [yxpan/children_sft_dataset](https://huggingface.co/datasets/yxpan/children_sft_dataset).

Instead of training one configuration at a time, we use RapidFire AI to run **4 configurations concurrently** and compare their training curves side-by-side in TensorBoard—all on a single free-tier Colab GPU.

## Our Approach

We use **Supervised Fine-Tuning (SFT)** with **LoRA (Low-Rank Adaptation)** to efficiently adapt TinyLlama for child-appropriate conversation. To find the best configuration, we vary **two axes** across **4 total runs**:

- **2 learning rates**: Conservative (`1e-5`) vs. aggressive (`1e-4`)—to study convergence stability vs. speed
- **2 LoRA adapter sizes**: Small (rank 8, Q/V projections only) vs. Large (rank 32, all attention projections Q/K/V/O)

The dataset uses **ChatML-style message formatting** with a system prompt that includes the child's age, ensuring the model learns to tailor its tone and vocabulary to different age groups.

RapidFire AI's **chunk-based scheduling** trains all 4 configurations concurrently—splitting the dataset into chunks and letting every run train on each chunk before moving to the next. This provides comparative metrics early, instead of waiting for full sequential training to finish.

### What You'll Learn

- How to define and run multiple SFT experiments concurrently with RapidFire AI
- Using ChatML-style message formatting with age-aware system prompts
- Using LoRA adapters of different capacities for parameter-efficient fine-tuning
- Evaluating child-facing chatbot quality with ROUGE-L and BERTScore metrics
- Monitoring all training curves simultaneously in TensorBoard
- Using Interactive Control Operations (IC Ops) to manage runs mid-training

## Install RapidFire AI Package and Setup

### Option 1: Install Locally (or on a VM)
For the full RapidFire AI experience—advanced experiment management, UI, and production features—we recommend installing the package on a machine you control (for example, a VM or your local machine) rather than Google Colab. See our [Install and Get Started](https://oss-docs.rapidfire.ai/en/latest/walkthrough.html) guide.

### Option 2: Install in Google Colab
For simplicity, you can run this notebook on Google Colab. The cell below installs `rapidfireai` if not already present, then runs `rapidfireai init` to set up the environment.

In [ ]:
try:
    import rapidfireai
    print("✅ rapidfireai already installed")
except ImportError:
    %pip install rapidfireai==0.14.0
    !rapidfireai init

## Start RapidFire Services

RapidFire AI runs three backend services: the **Dispatcher** (REST API on port 8851), **MLflow** (experiment tracking on port 8852), and the **Frontend** dashboard (port 8853). The cell below launches these services in the background.

- If any issues arise, check the status in a terminal window using `rapidfireai status` or `rapidfireai doctor`.
- You can also run `rapidfireai start` from a terminal on Colab instead of the cell below.

In [ ]:
import subprocess
from time import sleep
subprocess.Popen(["rapidfireai", "start"])
# sleep(30)

## Import RapidFire Components

We import the core RapidFire classes:
- **`Experiment`**: Top-level container that groups runs, saves artifacts, and tracks metrics.
- **`RFGridSearch`**: Generates all combinations of your knobs into a Config Group.
- **`RFModelConfig`**: Wraps the base model, LoRA config, and training arguments into a single unit.
- **`RFLoraConfig` / `RFSFTConfig`**: RapidFire wrappers around PEFT's `LoraConfig` and TRL's `SFTConfig`, enabling grid-searchable parameters via `List()`.

In [ ]:
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

## Load & Prepare Dataset

We use the [children_sft_dataset](https://huggingface.co/datasets/yxpan/children_sft_dataset), a custom dataset generated using the Gemini API. Each example contains:

- **`instruction`**: A child's question or statement
- **`response`**: An age-appropriate, supportive assistant reply
- **`age`**: The child's age group, used to tailor the system prompt

The dataset is split into **500 training** and **100 evaluation** examples to fit within free-tier Colab memory. For production training, increase these ranges significantly.

In [ ]:
from datasets import load_dataset

ds = load_dataset("yxpan/children_sft_dataset")

In [ ]:
train_dataset = ds['train'].select(range(500))    # !! make sure it does not exceed GPU memory constraints
eval_dataset = ds['train'].select(range(500,600))
train_dataset=train_dataset.shuffle(seed=42)
eval_dataset=eval_dataset.shuffle(seed=42)

## Inspect a Training Example

Let's look at a single example to understand the dataset structure before defining the formatting function.

In [ ]:
train_dataset[0]

## Define Data Formatting Function

SFT training with chat models requires **ChatML-style message formatting**. Each example is converted into a list of messages with `system`, `user`, and `assistant` roles. The system prompt dynamically includes the child's age so the model learns to adjust its tone and vocabulary accordingly.

This formatting function is passed to `RFModelConfig` and applied automatically during training.

In [ ]:
def sample_formatting_function(row):
    """Function to preprocess each example from dataset"""

    system_content = (
        f"You are talking to a child aged {row['age']}. "
        f"Generate a friendly response with age-appropriate knowledge."
    )


    # Standard ChatML-style dictionary
    return {
        "messages": [
            {"role": "system", "content": system_content},
            {"role": "user", "content": row['instruction']},
            {"role": "assistant", "content": row['response']}
        ]
    }



## Verify Formatting Output

Let's verify the formatting function produces the expected ChatML structure.

In [ ]:
sample_formatting_function(eval_dataset[0])


## Define Evaluation Metrics

We define two text quality metrics to evaluate generated responses against reference answers:

- **ROUGE-L**: Measures the longest common subsequence between prediction and reference, capturing structural similarity and fluency.
- **BERTScore**: Uses contextual BERT embeddings to compute semantic similarity between prediction and reference—more robust than n-gram metrics for evaluating conversational quality.

Both metrics are computed during evaluation steps and logged to TensorBoard.

In [ ]:
!pip install bert_score

In [ ]:
def sample_compute_metrics(eval_preds):
    import evaluate
    import numpy as np

    predictions, labels = eval_preds

    rouge = evaluate.load("rouge")
    bertscore = evaluate.load("bertscore")

    rouge_results = rouge.compute(predictions=predictions, references=labels)
    bert_results = bertscore.compute(predictions=predictions, references=labels, lang="en")

    return {
        "rougeL": round(rouge_results["rougeL"], 4),
        "bert_f1": round(np.mean(bert_results["f1"]), 4),
    }

## Initialize Experiment

Every RapidFire AI experiment needs a unique name. The `Experiment` object is the top-level container that groups all runs, saves artifacts, and tracks metrics. It automatically creates an MLflow experiment under the hood for structured logging.

In [ ]:
my_experiment = 'sft-child-age'
experiment = Experiment(experiment_name=my_experiment, mode="fit")

## Configure TensorBoard

We use TensorBoard to visualize training loss, evaluation metrics, and text quality scores across all 4 runs simultaneously. Setting the `RF_TRACKING_BACKEND` environment variable tells RapidFire to log all metrics to TensorBoard, making every run visible, comparable, and persistent.

In [ ]:
import os

# Load TensorBoard extension
%load_ext tensorboard

# Configure RapidFire to use TensorBoard
os.environ['RF_TRACKING_BACKEND'] = 'tensorboard'  # Options: 'mlflow', 'tensorboard', 'both'
# TensorBoard log directory will be auto-created in experiment path

In [ ]:
# Get experiment path
from rapidfireai.fit.db.rf_db import RfDb

db = RfDb()
experiment_path = db.get_experiments_path(my_experiment)
tensorboard_log_dir = f"{experiment_path}/tensorboard_logs/{my_experiment}"

print(f"TensorBoard logs will be saved to: {tensorboard_log_dir}")

## Define Experiment Configurations

This is where RapidFire AI shines. Instead of hardcoding a single training configuration, we define a search space and let RapidFire run all combinations concurrently.

### LoRA Primer

**LoRA (Low-Rank Adaptation)** adds small trainable matrices to frozen model layers instead of updating all weights, drastically reducing memory usage. Key parameters:

- **`r` (rank)**: Dimensionality of adapter matrices. We test rank 8 (lightweight) vs. rank 32 (higher capacity).
- **`lora_alpha`**: Scaling factor, typically proportional to rank.
- **`target_modules`**: Which layers to adapt. We compare Q/V projections only vs. all attention projections (Q/K/V/O).

### Configuration Matrix

We create 2 `RFModelConfig` entries (2 learning rates), each with the `peft_configs_lite` `List()` containing 2 LoRA configs. `RFGridSearch` expands this into **2 × 2 = 4 total runs**:

| Run | Learning Rate | LoRA Rank | Target Modules |
|-----|--------------|-----------|----------------|
| 1 | 1e-5 | 8 | q_proj, v_proj |
| 2 | 1e-5 | 32 | q_proj, k_proj, v_proj, o_proj |
| 3 | 1e-4 | 8 | q_proj, v_proj |
| 4 | 1e-4 | 32 | q_proj, k_proj, v_proj, o_proj |

In [ ]:
# 2 LoRA PEFT configs lite with different adapter capacities
peft_configs_lite = List([
    RFLoraConfig(
        r=8,
        lora_alpha=4,
        lora_dropout=0.1,
        target_modules=["q_proj", "v_proj"],  # Standard transformer naming
        bias="none"
    ),
    RFLoraConfig(
        r=32,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Standard naming
        bias="none"
    )
])

# 2 learning rates x 2 LoRA configs = 4 combinations in total
config_set_lite = List([
    RFModelConfig(
        model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",  # 1.1B model
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=1e-5,  # Higher LR for very small model
            lr_scheduler_type="linear",
            per_device_train_batch_size=4,
            per_device_eval_batch_size=4,
            max_steps=256,
            gradient_accumulation_steps=1,   # No accumulation needed
            logging_steps=2,
            eval_strategy="steps",
            eval_steps=20,
            fp16=True,
            # report_to="tensorboard",
        ),
        model_type="causal_lm",
        model_kwargs={"device_map": "auto", "torch_dtype": "auto", "use_cache": False},
        formatting_func=sample_formatting_function,
        compute_metrics=sample_compute_metrics,
        generation_config={
            "max_new_tokens": 128,
            "temperature": 0.8,  # Higher temp for tiny model
            "top_p": 0.9,
            "top_k": 30,         # Reduced top_k
            "repetition_penalty": 1.05,
        }
    ),
    RFModelConfig(
        model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",  # 1.1B model
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=1e-4,  # Higher LR for very small model
            lr_scheduler_type="linear",
            per_device_train_batch_size=4,  # Larger batch size
            per_device_eval_batch_size=4,
            max_steps=256,
            gradient_accumulation_steps=1,   # No accumulation needed
            logging_steps=2,
            eval_strategy="steps",
            eval_steps=20,
            fp16=True,
            # report_to="tensorboard",
        ),
        model_type="causal_lm",
        model_kwargs={"device_map": "auto", "torch_dtype": "auto", "use_cache": False},
        formatting_func=sample_formatting_function,
        compute_metrics=sample_compute_metrics,
        generation_config={
            "max_new_tokens": 128,
            "temperature": 0.8,  # Higher temp for tiny model
            "top_p": 0.9,
            "top_k": 30,         # Reduced top_k
            "repetition_penalty": 1.05,
        }
    ),

])

## Define Model Creation Function

RapidFire AI calls this function once per run to instantiate the model and tokenizer from the config dictionary. It receives the expanded config (with the specific LoRA/training args for that run) and must return a `(model, tokenizer)` tuple.

The function supports multiple model architectures (`causal_lm`, `seq2seq_lm`, `masked_lm`) for flexibility, though this experiment uses `causal_lm` (TinyLlama).

In [ ]:
# create model
def sample_create_model(model_config):
     """Function to create model object for any given config; must return tuple of (model, tokenizer)"""
     from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForMaskedLM

     model_name = model_config["model_name"]
     model_type = model_config["model_type"]
     model_kwargs = model_config["model_kwargs"]

     if model_type == "causal_lm":
          model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "seq2seq_lm":
          model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "masked_lm":
          model = AutoModelForMaskedLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "custom":
          # Handle custom model loading logic, e.g., loading your own checkpoints
          # model = ...
          pass
     else:
          # Default to causal LM
          model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

     tokenizer = AutoTokenizer.from_pretrained(model_name)

     return (model,tokenizer)

## Create Config Group

`RFGridSearch` takes the config set and generates a **Config Group**—the full Cartesian product of all `List()` knobs. Here, 2 learning rates × 2 LoRA configs = **4 concurrent runs**.

In [ ]:
# Grid search across all 4 config combinations
config_group = RFGridSearch(
    configs=config_set_lite,
    trainer_type="SFT"
)

## Start TensorBoard

**IMPORTANT: Start TensorBoard BEFORE running `run_fit()` below so you can watch metrics appear in real-time.**

Metrics appear gradually as each run completes training chunks. Seeing only a few runs at first is normal—all 4 will appear once their training begins.

In [ ]:
%tensorboard --logdir {tensorboard_log_dir}

## Run Training + Validation

Now we launch `run_fit()`—the main function for running multi-config training and evaluation. RapidFire AI will:

1. **Expand configs** into 4 individual runs (one per learning rate × LoRA combination).
2. **Split the dataset** into `num_chunks=4` chunks for interleaved execution.
3. **Schedule all runs** on each chunk before moving to the next, so you get comparative metrics after the very first chunk.
4. **Log metrics** to TensorBoard in real time—scroll up to watch training loss and eval scores update live.

`num_chunks` controls the swap granularity: more chunks = more frequent comparisons but slightly more overhead from model swapping.

In [ ]:
# Launch training of all configs in the config_group with swap granularity of 4 chunks
experiment.run_fit(config_group, sample_create_model, train_dataset, eval_dataset, num_chunks=4, seed=42)

## Launch Interactive Run Controller

RapidFire AI provides an Interactive Controller that lets you manage executing runs dynamically in real-time from the notebook:

- ⏹️ **Stop**: Gracefully stop a running config
- ▶️ **Resume**: Resume a stopped run
- 🗑️ **Delete**: Remove a run from this experiment
- 📋 **Clone**: Create a new run by editing the config dictionary of a parent run; optional warm start
- 🔄 **Refresh**: Update run status and metrics

In [ ]:
# Create Interactive Controller
sleep(15)
from rapidfireai.fit.utils.interactive_controller import InteractiveController

controller = InteractiveController(dispatcher_url="http://127.0.0.1:8851")
controller.display()

## End Experiment

Always end the experiment to finalize logging, save all artifacts, and clean up resources. Click the button below when you're ready to wrap up.

In [ ]:
from google.colab import output
from IPython.display import display, HTML

display(HTML('''
<button id="continue-btn" style="padding: 10px 20px; font-size: 16px;">Click to End Experiment</button>
'''))

# eval_js blocks until the Promise resolves
output.eval_js('''
new Promise((resolve) => {
    document.getElementById("continue-btn").onclick = () => {
        document.getElementById("continue-btn").disabled = true;
        document.getElementById("continue-btn").innerText = "Continuing...";
        resolve("clicked");
    };
})
''')

# Actually end the experiment after the button is clicked
experiment.end()
print("Done!")

## View TensorBoard Plots and Logs

After your experiment has ended, you can still view the full logs in TensorBoard:

In [ ]:
# View final logs
%tensorboard --logdir {tensorboard_log_dir}

## View RapidFire AI Log Files

RapidFire AI produces structured log files for each experiment. These are useful for debugging, auditing training behavior, or understanding what the scheduler and workers did under the hood.

In [ ]:
# Get the experiment-specific log file
from IPython.display import display, Pretty
log_file = experiment.get_log_file_path()

display(Pretty(f"📄 Experiment Log File: {log_file}"))

if log_file.exists():
    display(Pretty("=" * 80))
    display(Pretty(f"Last 30 lines of {log_file.name}:"))
    display(Pretty("=" * 80))
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            display(Pretty(line.rstrip()))
else:
    display(Pretty(f"❌ Log file not found: {log_file}"))

In [ ]:
# Get the training-specific log file
log_file = experiment.get_log_file_path("training")

display(Pretty(f"📄 Training Log File: {log_file}"))

if log_file.exists():
    display(Pretty("=" * 80))
    display(Pretty(f"Last 30 lines of {log_file.name}:"))
    display(Pretty("=" * 80))
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            display(Pretty(line.rstrip()))
else:
    display(Pretty(f"❌ Log file not found: {log_file}"))

## Conclusion

In this notebook, we fine-tuned a child-facing conversational chatbot by running **4 SFT configurations concurrently** on a single free-tier Colab GPU using RapidFire AI. Here's what we covered:

1. **Built a custom dataset pipeline** using Gemini-generated, age-aware instruction-response pairs with ChatML formatting.
2. **Defined a Config Group** using `List()` wrappers and `RFGridSearch` to compare learning rates and LoRA adapter sizes.
3. **Ran all configurations concurrently** with `run_fit()`, getting comparative metrics after each data chunk.
4. **Monitored training in real time** via TensorBoard, comparing loss curves, ROUGE-L, and BERTScore side-by-side.
5. **Used IC Ops** to manage runs mid-training—stopping underperformers or cloning promising configurations.

### Next Steps

- **Try different models**: Replace TinyLlama with larger models like [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), [Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B), or [Phi-3-mini](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct).
- **Expand the grid**: Add more learning rates, LoRA ranks, or target module combinations to explore a wider hyperparameter space.
- **Scale up the dataset**: Increase from 500 training samples to 5K+ for production-quality fine-tuning on a local machine or VM.
- **Add safety filters**: For production child-facing chatbots, integrate content safety layers and age-appropriate response guardrails.
- **Explore other training methods**: Use `RFDPOConfig` or `RFGRPOConfig` for preference alignment (DPO/GRPO) instead of SFT.
- **Use the RapidFire UI**: For a richer monitoring experience beyond TensorBoard, run RapidFire locally with the full dashboard at `http://localhost:8853`.

For more details, see the [RapidFire AI documentation](https://oss-docs.rapidfire.ai).